# AstroTutor — Allineamento DPO (QLoRA + TRL)

**Fase 5 del progetto**: allinea `Qwen2.5-3B-Instruct` sul dataset RAG-Aware generato da `src/alignment.py`, per scolpire nel modello la policy: *ancorati al contesto recuperato, adatta il registro al livello utente, rifiuta se il contesto non contiene la risposta*.

**Prerequisiti**
- Notebook Kaggle con acceleratore **GPU T4 x2** (Settings → Accelerator → GPU T4 x2; il codice usa una sola T4, che per un 3B in QLoRA è già sufficiente — non serve la seconda).
- Il file `data/alignment_data.json` caricato come **Dataset Kaggle** ("+ Add Input" in alto a destra → Upload → seleziona il file). Con ~400 triplette il training su una T4 dura indicativamente 1,5–3 ore (contro i 30–60 min stimati su A100): la T4 (Turing) non ha tensor core bf16 nativi ed è più lenta, ma rientra nel limite di sessione Kaggle.

**Pipeline**: dataset → Qwen2.5-3B in 4-bit (NF4) → adattatori LoRA → `DPOTrainer` (il modello di riferimento è implicito: gli adapter disattivati) → merge in fp16 → conversione GGUF `Q4_K_M` → `Modelfile` per Ollama.

**Output finale**: `astrotutor-3b-dpo-Q4_K_M.gguf` + `Modelfile`, salvati in `/kaggle/working/` (scaricabili dalla tab **Output** del notebook a fine esecuzione) — da installare in locale con `ollama create astrotutor-dpo -f Modelfile`.

In [ ]:
# 1) Dipendenze
!pip install -q -U trl peft bitsandbytes datasets accelerate transformers sentencepiece

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # riduce la frammentazione VRAM

import torch
assert torch.cuda.is_available(), "Attiva l'acceleratore GPU T4 x2 (Settings -> Accelerator)!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU disponibili: {torch.cuda.device_count()} (il notebook ne usa solo una)")

In [ ]:
# 2) Carica il dataset delle triplette
# Su Kaggle: carica data/alignment_data.json come Dataset del notebook
# ("+ Add Input" in alto a destra -> Upload -> seleziona il file), poi
# esegui questa cella: lo cerca automaticamente sotto /kaggle/input/.
import os
from pathlib import Path

DATA_FILE = ""  # opzionale: incolla qui il path esatto se preferisci non cercarlo

if not DATA_FILE:
    candidates = list(Path("/kaggle/input").rglob("alignment_data.json"))
    if not candidates:
        raise FileNotFoundError(
            "alignment_data.json non trovato in /kaggle/input/. "
            "Aggiungilo al notebook con '+ Add Input' -> Upload, poi rilancia questa cella."
        )
    DATA_FILE = str(candidates[0])

print(f"Dataset: {DATA_FILE} ({os.path.getsize(DATA_FILE)/1024:.0f} KB)")

In [ ]:
# 3) Dataset: solo le colonne DPO + split train/eval
from datasets import load_dataset

raw = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Triplette totali: {len(raw)}")
print("Distribuzione strategie:", sorted(set(raw["strategy"])))

dataset = raw.select_columns(["prompt", "chosen", "rejected"])
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} — Eval: {len(eval_ds)}")

# Sanity check sul formato conversazionale
example = train_ds[0]
assert isinstance(example["prompt"], list) and example["prompt"][0]["role"] == "system"
assert example["chosen"][0]["role"] == "assistant"
print("\nEsempio domanda:", example["prompt"][-1]["content"][:120])

In [ ]:
# 4) Modello base in 4-bit (QLoRA) + configurazione LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # stesso modello di qwen2.5:3b su Ollama

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # l'ambiente Kaggle forza bf16 (accelerate lo
                                             # sceglie comunque); la T4 non ha tensor core
                                             # bf16 ma esegue i kernel via CUDA core, solo
                                             # senza l'accelerazione dedicata
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
# 6) Salva l'adapter LoRA (backup leggero, ~100 MB)
# Su Kaggle basta scriverlo nella working directory: comparirà nella tab
# "Output" del notebook a fine esecuzione, scaricabile da lì.
trainer.save_model("astrotutor-dpo-adapter")
tokenizer.save_pretrained("astrotutor-dpo-adapter")

!zip -qr astrotutor-dpo-adapter.zip astrotutor-dpo-adapter
print("Salvato: astrotutor-dpo-adapter.zip (disponibile in Output a fine run)")

In [ ]:
# 7) Merge dell'adapter nel modello base (bf16, su CPU per liberare VRAM)
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, device_map="cpu"
)
merged = PeftModel.from_pretrained(base, "astrotutor-dpo-adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("astrotutor-3b-dpo-merged")
tokenizer.save_pretrained("astrotutor-3b-dpo-merged")
print("Merge completato: astrotutor-3b-dpo-merged/")

In [ ]:
# 8) Conversione in GGUF + quantizzazione Q4_K_M (per la RTX 3050 da 4 GB)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!python llama.cpp/convert_hf_to_gguf.py astrotutor-3b-dpo-merged \
    --outfile astrotutor-3b-dpo-f16.gguf --outtype f16

!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j > /dev/null
!llama.cpp/build/bin/llama-quantize \
    astrotutor-3b-dpo-f16.gguf astrotutor-3b-dpo-Q4_K_M.gguf Q4_K_M

!ls -lh astrotutor-3b-dpo-Q4_K_M.gguf

In [ ]:
# 9) Modelfile per Ollama (salvato in /kaggle/working/, scaricabile da Output)
modelfile = '''FROM ./astrotutor-3b-dpo-Q4_K_M.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
'''
with open("Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile scritto. A fine esecuzione, scarica dalla tab Output:")
print("  - astrotutor-3b-dpo-Q4_K_M.gguf")
print("  - Modelfile")

## Installazione in locale

1. Metti `astrotutor-3b-dpo-Q4_K_M.gguf` e `Modelfile` nella stessa cartella e crea il modello:
   ```
   ollama create astrotutor-dpo -f Modelfile
   ```
2. In `config.py` imposta:
   ```python
   LLM_MODEL = "astrotutor-dpo"
   ```
3. Verifica con `python main.py` (o `python compare.py` per l'A/B test contro la baseline).

**Valutazione consigliata** (Fase 6): stesse domande a `qwen2.5:3b` e `astrotutor-dpo`, confrontando (a) aderenza al registro del livello (Gulpease + LLM-judge), (b) fedeltà al contesto (RAGAS faithfulness), (c) tasso di rifiuto corretto sulle domande OOD.